# Tensor Decomposition
*Thomas Dooms*

In tutorial 1, we decomposed the bilinear interaction tensor using eigendecomposition. This gives us interpretable features *per output class*, but there are some issues with this. The most important one is orthogonality; you might have overlapping structures in input space which behave differently. The second is somewhat of a consequence where eigendecompositions often yield "superposed" features; we might hope that a 6 decomposes into a few edge-detectors but that generally doesn't happen. If the model shares structure across classes, eigendecomposition can't see it.

Here we take a different approach. Instead of decomposing per class, we decompose the *full* third-order tensor into a set of shared **neurons**, each with explicit input patterns and output weights. The result is a set of interpretable building blocks that the model combines to classify digits.

### Setup

In [1]:
!git clone https://github.com/shengweiming/bilinear-decomposition
%cd bilinear-decomposition
!pip install -e .

Cloning into 'bilinear-decomposition'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 157 (delta 13), reused 11 (delta 11), pack-reused 132 (from 1)
Receiving objects: 100% (157/157), 17.79 MiB | 17.37 MiB/s, done.
Resolving deltas: 100% (52/52), done.
/content/bilinear-decomposition
Obtaining file:///content/bilinear-decomposition
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.5/272.5 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import torch
import plotly.express as px
from plotly.subplots import make_subplots

from image import Model, MNIST
from image.sparse import Model as Sparse
from kornia.augmentation import RandomGaussianNoise

device = "cuda:0"
train, test = MNIST(train=True, device=device), MNIST(train=False, device=device)

100%|██████████| 9.91M/9.91M [00:00<00:00, 11.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 353kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.28MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.6MB/s]


### Training the original model

As in tutorial 1, we train a bilinear model with Gaussian noise augmentation to prevent overfitting on individual pixels.

In [15]:
model = Model.from_config(epochs=20).to(device)
model.fit(train, test, RandomGaussianNoise(std=0.4))

train/loss: 0.145, train/acc: 0.958, val/loss: 0.110, val/acc: 0.969: 100%|██████████| 20/20 [00:16<00:00,  1.19it/s]


,train/loss,train/acc,val/loss,val/acc
0,1.132581,0.687517,0.446589,0.8836
1,0.453151,0.872559,0.334491,0.9104
2,0.386020,0.891669,0.282399,0.9237
3,0.343214,0.901451,0.247095,0.9311
4,0.307735,0.913423,0.206934,0.9413
5,0.259505,0.925394,0.183544,0.9487
6,0.231982,0.931944,0.161961,0.9553
7,0.219108,0.936742,0.148345,0.9598
8,0.197924,0.942854,0.137090,0.9614
9,0.185841,0.946356,0.131954,0.9627


### From eigendecomposition to tensor decomposition

Recall that the bilinear model computes $\text{output}_c = \sum_{i,j} B_{c,i,j}\, x_i\, x_j$, where $B$ is the third-order interaction tensor. Eigendecomposition slices $B$ along the class axis and decomposes each slice independently.

We can instead factorize the *entire* tensor at once into a new bilinear layer:
$$B_{c,i,j} \approx \sum_{r=1}^{R} L_{i,r}\, R_{j,r}\, D_{c,r}$$

Each component $r$ is a **neuron** with three parts:
- $L_r \in \mathbb{R}^{784}$, the left input pattern
- $R_r \in \mathbb{R}^{784}$, the right input pattern  
- $D_r \in \mathbb{R}^{10}$, the output weights over classes

The neuron's activation on input $x$ is $(L_r^\top x)(R_r^\top x)$. Using the identity $ab = \tfrac{1}{4}(a+b)^2 - \tfrac{1}{4}(a-b)^2$, this becomes:
$$\tfrac{1}{4}\big((L_r + R_r)^\top x\big)^2 - \tfrac{1}{4}\big((L_r - R_r)^\top x\big)^2$$

So each neuron naturally decomposes into a **positive pattern** $L_r + R_r$ (matching it increases activation) and a **negative pattern** $L_r - R_r$ (matching it decreases activation). These are directly analogous to eigenvectors with positive and negative eigenvalues.

### Fitting the decomposition

We fit a new bilinear layer by maximizing the cosine similarity between the original interaction tensor and our approximation. The `Sparse` model stores $L$, $R$, and $D$ as learnable parameters and is optimized with Muon.

In [16]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

sparse = Sparse.from_config(rank=64).to(device)

optimizer = torch.optim.Muon(sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    loss = 1 - sparse.similarity(model)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"loss: {loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    orig_acc = (model(test.x).argmax(-1) == test.y).float().mean()
    sparse_acc = (sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"Original: {orig_acc:.3f}, Sparse: {sparse_acc:.3f}")

loss: 0.0019: 100%|██████████| 200/200 [00:01<00:00, 108.28it/s]

Original: 0.969, Sparse: 0.968


### Visualizing neurons

Each neuron has a positive pattern ($L_r + R_r$), a negative pattern ($L_r - R_r$), and output weights ($D_r$). The `decompose` method returns these normalized and sorted by importance (the product of the factor norms).

In [17]:
plus, minus, down, sigma = sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")

In [18]:
px.bar(sigma, template="plotly_white", labels=dict(index="component", value="sigma"))

### Exercises

The goal of this part is to implement a decomposition that is more in line with our requirements. This is where you try to find what kinds of priors and structure work well. The code above provides a skeleton with the essentials: the `Sparse` model, the reconstruction loop, and the visualization. There's no right answer and this is a hard task to get right.



List of tricks I tried


*   Add L1 regularization to decomp fitting for left and right matrices.

**Not impressive**




In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

sparse = Sparse.from_config(rank=64).to(device)

optimizer = torch.optim.Muon(sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

l1_coeff = 1  # tune this

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    recon_loss = 1 - sparse.similarity(model)
    l1_loss = sparse.left.abs().mean() + sparse.right.abs().mean()
    loss = recon_loss + l1_coeff * l1_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"recon: {recon_loss.item():.4f}, l1: {l1_loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    orig_acc = (model(test.x).argmax(-1) == test.y).float().mean()
    sparse_acc = (sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"Original: {orig_acc:.3f}, Sparse: {sparse_acc:.3f}")

recon: 0.0040, l1: 0.0283: 100%|██████████| 200/200 [00:01<00:00, 102.30it/s]


Original: 0.969, Sparse: 0.968


In [ ]:
plus, minus, down, sigma = sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")



*   Blow up rank

**Not impressive**




In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

sparse = Sparse.from_config(rank=512).to(device)

optimizer = torch.optim.Muon(sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

l1_coeff = 0.1  # tune this

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    recon_loss = 1 - sparse.similarity(model)
    l1_loss = (sparse.left.abs().mean()
           + sparse.right.abs().mean()
           + 1000 * sparse.down.abs().mean())  # weight `down` more
    loss = recon_loss + 0.01 * l1_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"recon: {recon_loss.item():.4f}, l1: {l1_loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    orig_acc = (model(test.x).argmax(-1) == test.y).float().mean()
    sparse_acc = (sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"Original: {orig_acc:.3f}, Sparse: {sparse_acc:.3f}")

recon: 0.0003, l1: 125.8101: 100%|██████████| 200/200 [00:03<00:00, 55.25it/s]

Original: 0.969, Sparse: 0.968


In [ ]:
plus, minus, down, sigma = sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")

In [ ]:
px.bar(sigma, template="plotly_white", labels=dict(index="component", value="sigma"))



*   Large l1 weight on the down matrix.

images = 2
sparsity = 4


In [6]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

sparse = Sparse.from_config(rank=32).to(device)

optimizer = torch.optim.Muon(sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

l1_coeff = 0.01  # tune this

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    recon_loss = 1 - sparse.similarity(model)
    l1_loss = (sparse.left.abs().mean()
           + sparse.right.abs().mean()
           + 100 * sparse.down.abs().mean())  # weight `down` more
    loss = recon_loss + 0.01 * l1_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"recon: {recon_loss.item():.4f}, l1: {l1_loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    orig_acc = (model(test.x).argmax(-1) == test.y).float().mean()
    sparse_acc = (sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"Original: {orig_acc:.3f}, Sparse: {sparse_acc:.3f}")

recon: 0.0157, l1: 3.9621: 100%|██████████| 200/200 [00:02<00:00, 70.22it/s] 

Original: 0.969, Sparse: 0.962


In [7]:
plus, minus, down, sigma = sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")

In [8]:
px.bar(sigma, template="plotly_white", labels=dict(index="component", value="sigma"))



*   Retrain model with more noise

images = 4

sparse = 3


In [12]:
model = Model.from_config(epochs=20).to(device)
model.fit(train, test, RandomGaussianNoise(std=0.8))

train/loss: 0.274, train/acc: 0.918, val/loss: 0.135, val/acc: 0.965: 100%|██████████| 20/20 [00:14<00:00,  1.39it/s]


,train/loss,train/acc,val/loss,val/acc
0,1.262634,0.629209,0.442404,0.8806
1,0.624593,0.818763,0.374346,0.9038
2,0.546912,0.838008,0.317973,0.9207
3,0.491732,0.854105,0.263196,0.9343
4,0.441920,0.870252,0.224483,0.9440
5,0.393169,0.882122,0.200286,0.9494
6,0.361774,0.892578,0.179143,0.9545
7,0.352510,0.895306,0.171107,0.9561
8,0.327587,0.901569,0.160811,0.9587
9,0.315254,0.905004,0.155696,0.9597


In [15]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

sparse = Sparse.from_config(rank=32).to(device)

optimizer = torch.optim.Muon(sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

l1_coeff = 0.01  # tune this

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    recon_loss = 1 - sparse.similarity(model)
    l1_loss = (sparse.left.abs().mean()
           + sparse.right.abs().mean()
           + 100 * sparse.down.abs().mean())  # weight `down` more
    loss = recon_loss + 0.01 * l1_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"recon: {recon_loss.item():.4f}, l1: {l1_loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    orig_acc = (model(test.x).argmax(-1) == test.y).float().mean()
    sparse_acc = (sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"Original: {orig_acc:.3f}, Sparse: {sparse_acc:.3f}")

recon: 0.0268, l1: 3.9347: 100%|██████████| 200/200 [00:01<00:00, 114.53it/s]


Original: 0.965, Sparse: 0.960


In [16]:
plus, minus, down, sigma = sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")



*   Blur, rotation, and other noise, also double model training time

images = 4

sparse = 2



In [19]:
from kornia.augmentation import RandomGaussianNoise, RandomRotation, RandomGaussianBlur, AugmentationSequential

model = Model.from_config(epochs=40).to(device)

augment = AugmentationSequential(
    RandomRotation(degrees=15.0, p=0.5),
    RandomGaussianBlur(kernel_size=(3, 3), sigma=(0.3, 0.7), p=0.5),
    RandomGaussianNoise(std=0.4, p=1.0),
)

model.fit(train, test, augment)

train/loss: 0.247, train/acc: 0.926, val/loss: 0.129, val/acc: 0.964:  25%|██▌       | 10/40 [00:09<00:27,  1.10it/s]


KeyboardInterrupt: 

In [34]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

sparse = Sparse.from_config(rank=32).to(device)

optimizer = torch.optim.Muon(sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

l1_coeff = 0.01  # tune this

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    recon_loss = 1 - sparse.similarity(model)
    l1_loss = (sparse.left.abs().mean()
           + sparse.right.abs().mean()
           + 100 * sparse.down.abs().mean())  # weight `down` more
    loss = recon_loss + 0.01 * l1_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"recon: {recon_loss.item():.4f}, l1: {l1_loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    orig_acc = (model(test.x).argmax(-1) == test.y).float().mean()
    sparse_acc = (sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"Original: {orig_acc:.3f}, Sparse: {sparse_acc:.3f}")

recon: 0.0318, l1: 4.3492: 100%|██████████| 200/200 [00:01<00:00, 104.89it/s]

Original: 0.977, Sparse: 0.961


In [35]:
plus, minus, down, sigma = sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")

* add l2 weight decay to the original model

**cleaner image + relatively sparse output**

In [23]:
from kornia.augmentation import RandomGaussianNoise, RandomRotation, RandomGaussianBlur, AugmentationSequential

model = Model.from_config(epochs=40, weight_decay=3.0).to(device)
augment = AugmentationSequential(
    RandomRotation(degrees=15.0, p=0.5),
    RandomGaussianBlur(kernel_size=(3, 3), sigma=(0.3, 0.7), p=0.5),
    RandomGaussianNoise(std=0.4, p=1.0),
)
model.fit(train, test, augment)

train/loss: 0.158, train/acc: 0.953, val/loss: 0.082, val/acc: 0.977: 100%|██████████| 40/40 [00:32<00:00,  1.25it/s]


,train/loss,train/acc,val/loss,val/acc
0,1.223110,0.640861,0.440107,0.8828
1,0.570239,0.833951,0.342498,0.9093
2,0.489119,0.853465,0.290367,0.9213
3,0.425546,0.873030,0.236479,0.9366
4,0.372559,0.890692,0.197416,0.9458
5,0.330619,0.902563,0.173711,0.9516
6,0.301715,0.911486,0.158600,0.9566
7,0.279591,0.917514,0.144173,0.9589
8,0.257568,0.922751,0.136494,0.9608
9,0.247015,0.925613,0.128584,0.9637


In [24]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

sparse = Sparse.from_config(rank=32).to(device)

optimizer = torch.optim.Muon(sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

l1_coeff = 0.01  # tune this

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    recon_loss = 1 - sparse.similarity(model)
    l1_loss = (sparse.left.abs().mean()
           + sparse.right.abs().mean()
           + 100 * sparse.down.abs().mean())  # weight `down` more
    loss = recon_loss + 0.01 * l1_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"recon: {recon_loss.item():.4f}, l1: {l1_loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    orig_acc = (model(test.x).argmax(-1) == test.y).float().mean()
    sparse_acc = (sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"Original: {orig_acc:.3f}, Sparse: {sparse_acc:.3f}")

recon: 0.0318, l1: 4.3492: 100%|██████████| 200/200 [00:01<00:00, 112.82it/s]


Original: 0.977, Sparse: 0.961


In [25]:
plus, minus, down, sigma = sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")



*   add l1 penalty to model

Found a 9! And a 7, kind of.

image = 5

sparsity = 3




In [60]:
model = Model.from_config(epochs=20, weight_decay = 0.5, l1 = 1e-4).to(device)
model.fit(train, test, RandomGaussianNoise(std=0.4))

train/loss: 0.528, train/acc: 0.921, val/loss: 0.223, val/acc: 0.939: 100%|██████████| 20/20 [00:12<00:00,  1.62it/s]


,train/loss,train/acc,val/loss,val/acc
0,1.856307,0.647309,0.451763,0.8835
1,1.002460,0.872424,0.350493,0.9062
2,0.877173,0.887527,0.325117,0.9158
3,0.812501,0.889918,0.318342,0.9140
4,0.769124,0.892528,0.303088,0.9176
5,0.719888,0.897478,0.293775,0.9200
6,0.688836,0.900290,0.284240,0.9236
7,0.668770,0.900845,0.272090,0.9276
8,0.639819,0.904751,0.264443,0.9275
9,0.620709,0.907479,0.257225,0.9296


In [61]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

sparse = Sparse.from_config(rank=16).to(device)

optimizer = torch.optim.Muon(sparse.parameters(), lr=0.02, momentum=0.95)
scheduler = CosineAnnealingLR(optimizer, T_max=200)

l1_coeff = 0.01  # tune this

torch.set_grad_enabled(True)
for _ in (pbar := tqdm(range(200))):
    recon_loss = 1 - sparse.similarity(model)
    l1_loss = (sparse.left.abs().mean()
           + sparse.right.abs().mean()
           + 150 * sparse.down.abs().mean())  # weight `down` more
    loss = recon_loss + 0.01 * l1_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    pbar.set_description(f"recon: {recon_loss.item():.4f}, l1: {l1_loss.item():.4f}")
torch.set_grad_enabled(False)

with torch.no_grad():
    orig_acc = (model(test.x).argmax(-1) == test.y).float().mean()
    sparse_acc = (sparse(test.x).argmax(-1) == test.y).float().mean()
    print(f"Original: {orig_acc:.3f}, Sparse: {sparse_acc:.3f}")

recon: 0.0073, l1: 2.6189: 100%|██████████| 200/200 [00:01<00:00, 125.83it/s]

Original: 0.939, Sparse: 0.934


In [62]:
plus, minus, down, sigma = sparse.decompose()
plus, minus, down, sigma = plus.cpu(), minus.cpu(), down.cpu(), sigma.cpu()

k = 8
fig = make_subplots(rows=3, cols=k, row_titles=["l+r", "l-r", "logits"], vertical_spacing=0.08)

for i in range(k):
    params = dict(showscale=False, colorscale="RdBu", zmid=0)
    fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i+1)
    fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i+1)
    fig.add_bar(y=down[:, i], marker_color=["gray"]*10, showlegend=False, row=3, col=i+1)

fig.update_xaxes(visible=False).update_yaxes(visible=False)
fig.update_layout(width=k*110, height=360, margin=dict(l=40, r=0, b=0, t=0), template="plotly_white")